In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Klasifikasi: Sebuah Qiskit Function oleh Multiverse Computing
> **Note:** * Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Fitur ini masih dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.

## Ikhtisar
Dengan fungsi "Singularity Machine Learning - Klasifikasi", kamu bisa memecahkan masalah machine learning di dunia nyata menggunakan hardware kuantum tanpa perlu keahlian kuantum. Application function ini, yang didasarkan pada metode ensemble, adalah classifier hybrid. Fungsi ini memanfaatkan metode klasik seperti boosting, bagging, dan stacking untuk pelatihan ensemble awal. Selanjutnya, algoritma kuantum seperti variational quantum eigensolver (VQE) dan quantum approximate optimization algorithm (QAOA) digunakan untuk meningkatkan keberagaman, kemampuan generalisasi, dan kompleksitas keseluruhan ensemble yang telah dilatih.

Tidak seperti solusi quantum machine learning lainnya, fungsi ini mampu menangani dataset berskala besar dengan jutaan contoh dan fitur tanpa dibatasi oleh jumlah qubit pada QPU target. Jumlah qubit hanya menentukan ukuran ensemble yang dapat dilatih. Fungsi ini juga sangat fleksibel dan dapat digunakan untuk memecahkan masalah klasifikasi di berbagai domain, termasuk keuangan, kesehatan, dan keamanan siber.
Fungsi ini secara konsisten mencapai akurasi tinggi pada masalah yang sulit secara klasik, melibatkan dataset berdimensi tinggi, berisik, dan tidak seimbang.
![Cara kerjanya](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Fungsi ini dibuat untuk:
1. Para engineer dan ilmuwan data di perusahaan yang ingin meningkatkan penawaran teknologi mereka dengan mengintegrasikan quantum machine learning ke dalam produk dan layanan mereka,
2. Para peneliti di laboratorium riset kuantum yang mengeksplorasi aplikasi quantum machine learning dan ingin memanfaatkan komputasi kuantum untuk tugas klasifikasi, serta
3. Mahasiswa dan pengajar di institusi pendidikan dalam mata kuliah seperti machine learning, yang ingin mendemonstrasikan keunggulan komputasi kuantum.

Contoh berikut menampilkan berbagai fungsinya, termasuk `create`, `list`, `fit`, dan `predict`, serta mendemonstrasikan penggunaannya pada masalah sintetis yang terdiri dari dua setengah lingkaran yang saling bersilangan — sebuah masalah yang terkenal sulit karena batas keputusannya yang nonlinier.


## Deskripsi fungsi
Qiskit Function ini memungkinkan pengguna memecahkan masalah klasifikasi biner menggunakan classifier ensemble yang ditingkatkan kuantum dari Singularity. Di balik layar, fungsi ini menggunakan pendekatan hybrid untuk melatih ensemble classifier secara klasik pada dataset berlabel, lalu mengoptimalkannya untuk keberagaman dan generalisasi maksimal menggunakan Quantum Approximate Optimization Algorithm (QAOA) pada QPU IBM&reg;. Melalui antarmuka yang ramah pengguna, pengguna dapat mengonfigurasi classifier sesuai kebutuhan mereka, melatihnya pada dataset pilihan, dan menggunakannya untuk membuat prediksi pada dataset yang belum pernah dilihat sebelumnya.

Untuk memecahkan masalah klasifikasi umum:
1. Praproses dataset, lalu bagi menjadi set pelatihan dan pengujian. Secara opsional, kamu bisa membagi lebih lanjut set pelatihan menjadi set pelatihan dan validasi. Hal ini dapat dilakukan menggunakan [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Jika set pelatihan tidak seimbang, kamu bisa melakukan resampling untuk menyeimbangkan kelas menggunakan [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Unggah set pelatihan, validasi, dan pengujian secara terpisah ke penyimpanan fungsi menggunakan metode `file_upload` dari catalog, dengan meneruskan path yang relevan setiap kali.
4. Inisialisasi classifier kuantum menggunakan aksi `create` dari fungsi, yang menerima hyperparameter seperti jumlah dan jenis learner, regularisasi (nilai lambda), serta opsi optimasi termasuk jumlah lapisan, jenis classical optimizer, quantum Backend, dan sebagainya.
5. Latih classifier kuantum pada set pelatihan menggunakan aksi `fit` dari fungsi, dengan meneruskan set pelatihan berlabel, dan set validasi jika ada.
6. Buat prediksi pada set pengujian yang belum pernah dilihat menggunakan aksi `predict` dari fungsi.
## Pendekatan berbasis aksi
Fungsi ini menggunakan pendekatan berbasis aksi. Kamu bisa membayangkannya sebagai lingkungan virtual di mana kamu menggunakan aksi untuk melakukan tugas atau mengubah statusnya. Saat ini, fungsi ini menawarkan aksi-aksi berikut: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict), dan [create_fit_predict](#7-create-fit-predict). Contoh berikut mendemonstrasikan aksi `create_fit_predict`.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List
Aksi `list` mengambil semua classifier yang tersimpan dalam format `*.pkl.tar` dari direktori data bersama. Kamu juga bisa mengakses isi direktori ini menggunakan metode `catalog.files()`. Secara umum, aksi list mencari file dengan ekstensi `*.pkl.tar` di direktori data bersama dan mengembalikannya dalam format list.
#### Masukan
|     Nama    |    Tipe     | Deskripsi |   Wajib  |
|-------------|-------------|-----------|----------|
| `action` | `str` | Nama aksi dari antara `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict`, dan `delete`. | Ya |

#### Penggunaan

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create
Aksi `create` membuat classifier dengan tipe `quantum_classifier` yang ditentukan menggunakan parameter yang diberikan, dan menyimpannya di direktori data bersama.

> **Note:** Fungsi ini saat ini hanya mendukung `QuantumEnhancedEnsembleClassifier`.
#### Masukan
|     Nama    |    Tipe     | Deskripsi | Wajib  | Default |
|-------------|-------------|-----------|--------|---------|
| `action` | `str` | Nama aksi dari antara `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict`, dan `delete`. | Ya | - |
| `name` | `str` | Nama classifier kuantum, misalnya `spam_classifier`. | Ya | - |
| `instance` | `str` | Instance IBM. | Ya | - |
| `backend_name` | `str` | Sumber daya komputasi IBM. Default-nya adalah `None`, yang berarti Backend dengan pekerjaan pending paling sedikit yang akan digunakan. | Tidak | `None` |
| `quantum_classifier` | `str` | Tipe classifier kuantum, yaitu `QuantumEnhancedEnsembleClassifier`. | Tidak | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Jumlah learner dalam ensemble. | Tidak | `10` |
| `learners_types` | `list` | Tipe-tipe learner. Di antara tipe yang didukung: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier`, dan `LogisticRegression`. Detail lebih lanjut terkait masing-masing dapat ditemukan di [dokumentasi scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Tidak | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporsi setiap tipe learner dalam ensemble. | Tidak | `[1.0]` |
| `learners_options` | `list` | Opsi untuk setiap tipe learner dalam ensemble. Untuk daftar lengkap opsi yang sesuai dengan tipe learner yang dipilih, lihat [dokumentasi scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Tidak | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` atau `list` | Tipe regularisasi yang digunakan: `onsite` atau `alpha`. `onsite` mengontrol suku onsite di mana nilai yang lebih tinggi menghasilkan ensemble yang lebih sparse. `alpha` mengontrol trade-off antara suku interaksi dan onsite di mana nilai yang lebih rendah menghasilkan ensemble yang lebih sparse. Jika list diberikan, model akan dilatih untuk setiap tipe dan yang berkinerja terbaik akan dipilih. | Tidak | `onsite` |
| `regularization` | `str` atau `float` atau `list` | Nilai regularisasi. Dibatasi antara `0` dan `+inf` jika regularization_type adalah `onsite`. Dibatasi antara `0` dan `1` jika regularization_type adalah `alpha`. Jika diatur ke `auto`, regularisasi otomatis digunakan — parameter regularisasi optimal ditemukan melalui binary search dengan rasio classifier yang dipilih terhadap total classifier yang diinginkan (`regularization_desired_ratio`) dan batas atas parameter regularisasi (`regularization_upper_bound`). Jika list diberikan, model akan dilatih untuk setiap nilai dan yang berkinerja terbaik akan dipilih. | Tidak | `0.01` |
| `regularization_desired_ratio` | `float` atau `list` | Rasio yang diinginkan dari classifier yang dipilih terhadap total classifier untuk regularisasi otomatis. Jika list diberikan, model akan dilatih untuk setiap rasio dan yang berkinerja terbaik akan dipilih. | Tidak | `0.75` |
| `regularization_upper_bound` | `float` atau `list` | Batas atas parameter regularisasi saat menggunakan regularisasi otomatis. Jika list diberikan, model akan dilatih untuk setiap batas atas dan yang berkinerja terbaik akan dipilih. | Tidak | `200` |
| `weight_update_method` | `str` | Metode pembaruan bobot sampel dari antara `logarithmic` dan `quadratic`. | Tidak | `logarithmic` |
| `sample_scaling` | `boolean` | Apakah penskalaan sampel harus diterapkan. | Tidak | `False` |
| `prediction_scaling` | `float` | Faktor penskalaan untuk prediksi. | Tidak | `None` |
| `optimizer_options` | `dictionary` | Opsi optimizer QAOA. Daftar opsi yang tersedia disajikan kemudian dalam dokumentasi ini. | Tidak | ... |
| `voting` | `str` | Gunakan majority voting (`hard`) atau rata-rata probabilitas (`soft`) untuk mengagregasi prediksi/probabilitas learner. | Tidak | `hard` |
| `prob_threshold` | `float` | Ambang batas probabilitas optimal. | Tidak | `0.5` |
| `random_state` | `integer` | Kendalikan keacakan untuk keterproduksian. | Tidak | `None` |


- Selain itu, `optimizer_options` dirinci sebagai berikut:

|     Nama    |    Tipe     | Deskripsi | Wajib  | Default |
|-------------|-------------|-----------|--------|---------|
| `num_solutions` | `integer` | Jumlah solusi | Tidak | `1024` |
| `reps` | `integer` | Jumlah pengulangan | Tidak | `4` |
| `sparsify` | `float` | Ambang batas sparsification | Tidak | `0.001` |
| `theta` | `float` | Nilai awal theta, parameter variasional QAOA | Tidak | `None` |
| `simulator` | `boolean` | Apakah menggunakan simulator atau QPU | Tidak | `False` |
| `classical_optimizer` | `str` | Nama classical optimizer untuk QAOA. Semua solver yang ditawarkan oleh SciPy, sebagaimana tercantum [di sini](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), dapat digunakan. Kamu perlu mengatur `classical_optimizer_options` sesuai kebutuhan | Tidak | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opsi classical optimizer. Untuk daftar lengkap opsi yang tersedia, lihat [dokumentasi SciPy](https://docs.scipy.org/doc/scipy/reference/) | Tidak | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Kedalaman Circuit QAOA | Tidak | `3` |
| `num_transpiler_runs` | `integer` | Jumlah jalankan Transpiler | Tidak | `30` |
| `pass_manager_options` | `dictionary` | Opsi untuk menghasilkan preset pass manager | Tidak | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opsi Estimator. Untuk daftar lengkap opsi yang tersedia, lihat [dokumentasi Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Tidak | `None` |
| `sampler_options` | `dictionary` | Opsi Sampler. Untuk daftar lengkap opsi yang tersedia, lihat [dokumentasi Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Tidak | `None` |

- Default `estimator_options` adalah:

|     Nama    |    Tipe     | Nilai  |
|-------------|-------------|--------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Default `sampler_options` adalah:

|     Nama    |    Tipe     | Nilai |
|-------------|-------------|-------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Penggunaan

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### Validasi
- `name`:
    - Nama harus unik, berupa string hingga 64 karakter.
    - Hanya boleh mengandung karakter alfanumerik dan garis bawah.
    - Harus dimulai dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Classifier dengan nama yang sama harus sudah ada di direktori data bersama.
### 4. Fit
Aksi `fit` melatih classifier menggunakan data pelatihan yang diberikan.

#### Masukan
|     Nama    |    Tipe     | Deskripsi |   Wajib  |
|-------------|-------------|-----------|----------|
| `action`    | `str`    | Nama aksi. Harus `fit`. | Ya |
| `name`      | `str`    | Nama classifier yang akan dilatih. | Ya |
| `X`         | `array` atau `list` atau `str` | Data pelatihan. Bisa berupa array NumPy, list, atau string yang merujuk nama file di direktori data bersama. | Ya |
| `y`         | `array` atau `list` atau `str` | Nilai target pelatihan. Bisa berupa array NumPy, list, atau string yang merujuk nama file di direktori data bersama. | Ya |
| `fit_params`| `dictionary`| Parameter tambahan untuk diteruskan ke metode `fit` dari classifier. | Tidak |

##### fit_params
|     Nama    |    Tipe     | Deskripsi |   Wajib  |   Default   |
|-------------|-------------|-----------|----------|-------------|
| `validation_data` | `tuple` | Data validasi dan labelnya. | Tidak | `None` |
| `pos_label` | `integer` atau `str` | Label kelas yang dipetakan ke 1. | Tidak | `None` |
| `optimization_data` | `str` | Dataset untuk mengoptimalkan ensemble. Bisa salah satu dari: `train`, `validation`, `both`. | Tidak | `train` |

#### Penggunaan

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### Validasi
- `name`:
    - Nama harus unik, berupa string hingga 64 karakter.
    - Hanya boleh mengandung karakter alfanumerik dan garis bawah.
    - Harus dimulai dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Classifier dengan nama yang sama harus sudah ada di direktori data bersama.
### 5. Predict
Aksi `predict` digunakan untuk mendapatkan prediksi keras dan lunak (probabilitas).

#### Input
|     Nama    |    Tipe     | Deskripsi |   Wajib  |
|-------------|-------------|-----------|----------|
| `action`    | `str`    | Nama aksi. Harus berupa `predict`. | Ya |
| `name`      | `str`    | Nama classifier yang akan digunakan. | Ya |
| `X`         | `array` atau `list` atau `str` | Data uji. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `options["out"]` | `str` | Nama file JSON keluaran untuk menyimpan prediksi di direktori data bersama. Jika tidak disediakan, prediksi dikembalikan dalam hasil job. | Tidak |

#### Penggunaan

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### Validasi
- `name`:
    - Nama harus unik, berupa string hingga 64 karakter.
    - Hanya boleh berisi karakter alfanumerik dan garis bawah.
    - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Classifier dengan nama yang sama harus sudah ada di direktori data bersama.
- `options["out"]`:
    - Nama file harus unik, berupa string hingga 64 karakter.
    - Hanya boleh berisi karakter alfanumerik dan garis bawah.
    - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Harus memiliki ekstensi `.json`.
### 6. Fit-predict
Aksi `fit_predict` melatih classifier menggunakan data pelatihan, lalu menggunakannya untuk mendapatkan prediksi keras dan lunak (probabilitas).

#### Input
|     Nama    |    Tipe     | Deskripsi |   Wajib  |
|-------------|-------------|-----------|----------|
| `action`    | `str`    | Nama aksi. Harus berupa `fit_predict`. | Ya |
| `name`      | `str`    | Nama classifier yang akan digunakan. | Ya |
| `X_train`   | `array` atau `list` atau `str` | Data pelatihan. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `y_train`   | `array` atau `list` atau `str` | Nilai target pelatihan. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `X_test`    | `array` atau `list` atau `str` | Data uji. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `fit_params`| `dictionary`| Parameter tambahan yang diteruskan ke metode `fit` classifier. | Tidak |
| `options["out"]` | `str` | Nama file JSON keluaran untuk menyimpan prediksi di direktori data bersama. Jika tidak disediakan, prediksi dikembalikan dalam hasil job. | Tidak |

#### Penggunaan

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### Validasi
- `name`:
    - Nama harus unik, berupa string hingga 64 karakter.
    - Hanya boleh berisi karakter alfanumerik dan garis bawah.
    - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Classifier dengan nama yang sama harus sudah ada di direktori data bersama.

- `options["out"]`:
    - Nama file harus unik, berupa string hingga 64 karakter.
    - Hanya boleh berisi karakter alfanumerik dan garis bawah.
    - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Harus memiliki ekstensi `.json`.
### 7. Create-fit-predict
Aksi `create_fit_predict` membuat classifier, melatihnya menggunakan data pelatihan yang diberikan, lalu menggunakannya untuk mendapatkan prediksi keras dan lunak (probabilitas).

#### Input
| Nama | Tipe | Deskripsi | Wajib |
|------|------|-----------|-------|
| `action` | `str` | Nama aksi dari antara `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict`, dan `delete`. | Ya |
| `name` | `str` | Nama classifier yang akan digunakan. | Ya |
| `quantum_classifier` | `str` | Tipe classifier, yaitu `QuantumEnhancedEnsembleClassifier`. Default-nya adalah `QuantumEnhancedEnsembleClassifier`. | Tidak |
| `X_train` | `array` atau `list` atau `str` | Data pelatihan. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `y_train` | `array` atau `list` atau `str` | Nilai target pelatihan. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `X_test` | `array` atau `list` atau `str` | Data uji. Bisa berupa NumPy array, list, atau string yang merujuk pada nama file di direktori data bersama. | Ya |
| `fit_params` | `dictionary` | Parameter tambahan yang diteruskan ke metode `fit` classifier. | Tidak |
| `options["save"]` | `boolean` | Apakah classifier yang sudah dilatih akan disimpan di direktori data bersama. Default-nya adalah `True`. | Tidak |
| `options["out"]` | `str` | Nama file JSON keluaran untuk menyimpan prediksi di direktori data bersama. Jika tidak disediakan, prediksi dikembalikan dalam hasil job. | Tidak |

#### Penggunaan

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### Validasi
- `name`:
    - Jika `options["save"]` diatur ke `True`:
        - Nama harus unik, berupa string hingga 64 karakter.
        - Hanya boleh berisi karakter alfanumerik dan garis bawah.
        - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
        - Tidak boleh ada classifier dengan nama yang sama di direktori data bersama.

- `options["out"]`:
    - Nama file harus unik, berupa string hingga 64 karakter.
    - Hanya boleh berisi karakter alfanumerik dan garis bawah.
    - Harus diawali dengan huruf dan tidak boleh diakhiri dengan garis bawah.
    - Harus memiliki ekstensi `.json`.
---
## Mulai
Autentikasi menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/) kamu, lalu pilih Qiskit Function seperti berikut:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Contoh
Dalam contoh ini, kamu akan menggunakan fungsi "Singularity Machine Learning - Classification" untuk mengklasifikasi dataset yang terdiri dari dua setengah lingkaran berbentuk bulan yang saling terkait. Dataset ini bersifat sintetis, dua dimensi, dan diberi label biner. Dataset ini dirancang agar menantang bagi algoritma seperti clustering berbasis centroid dan klasifikasi linear.
![Dataset moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Melalui proses ini, kamu akan belajar cara membuat classifier, melatihnya pada data pelatihan, menggunakannya untuk memprediksi data uji, dan menghapus classifier setelah selesai.
Sebelum mulai, kamu perlu menginstal [scikit-learn](https://scikit-learn.org/). Instal menggunakan perintah berikut:
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).